# You can train a transformer! 🎲 — Make a Choice edition

### 🛑 STOP! Switch to a GPU first!

**Runtime → Change runtime type → T4 GPU → Save.**

Click ▶ next to a cell, or press **Shift + Enter**. Try these two cells first!


In [ ]:
print('Hello, world!')


**And this one!**


In [ ]:
import torch
assert torch.cuda.is_available(), '🛑 Switch to T4 GPU: Runtime → Change runtime type → T4 GPU → Save.'
print('GPU:', torch.cuda.get_device_name(0))


Nice! Keep running cells from top to bottom. **Don't click Run all**: you'll choose **stories** or **math** at *Choose your adventure*.

We're fine-tuning GPT-2: really changing the model's weights.


## 1. Bring the tools 🧰

We install two libraries. The rest is ordinary Python and PyTorch, which Colab already provides.

**A couple of warnings you might see:** Colab sometimes prints a `pip dependency conflicts` message mentioning preinstalled packages like `diffusers` or `gradio`. This notebook doesn't use them, and the warning alone doesn't mean our installation failed. When GPT-2 downloads, Hugging Face may also say `HF_TOKEN` is missing; **no token or Hugging Face account is needed for this public model and dataset**. If a cell actually stops with an error, that's different—check the message before continuing.

In [ ]:
%pip -q install "transformers>=4.44,<5" "datasets>=3.0,<5"

## 2. Meet GPT-2 🤖

This is the original small GPT-2 (124 million parameters). We'll save its initial text so we can compare it after training.

In [ ]:
import random
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

random.seed(42)
torch.manual_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Training on:", DEVICE)
if DEVICE == "cpu":
    print("⚠️ This will work on CPU, but training may be slow. A Colab T4 GPU is recommended.")

MODEL_NAME = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)

def try_prompt(prompt):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        ids = model.generate(**inputs, max_new_tokens=60, do_sample=True,
                             temperature=0.8, top_p=0.95,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(ids[0], skip_special_tokens=True)

def read_math_answer(output):
    """Decode the model's three reverse-order digit tokens into a normal number."""
    match = re.fullmatch(r"\s*([0-9])\s*([0-9])\s*([0-9])\s*", output)
    if match is None:
        return None
    return int("".join(match.groups())[::-1])


# Fixed story openings for our before/after comparisons; don't change them between runs!
STORY_PROMPTS = [
    "Once upon a time, there was a little dragon",
    "Lily found a shiny key under her bed",
    "A tiny robot wanted to make a new friend",
]
# These exact addition problems are in the held-out test split, not training.
MATH_PROBE_PAIRS = [(74, 88), (92, 52), (57, 14)]
MATH_PROMPTS = [
    f"{a // 10} {a % 10} + {b // 10} {b % 10} ="
    for a, b in MATH_PROBE_PAIRS
]

@torch.no_grad()
def show_examples(heading, show_stories=True, show_math=True):
    """Revisit just the story or math prompts the reader wants to compare."""
    model.eval()
    print("\n" + "=" * 16 + " " + heading + " " + "=" * 16)
    if show_stories:
        print("\n📖 THREE STORIES (same openings and random seeds each time):")
        for index, prompt in enumerate(STORY_PROMPTS):
            # Independent seeds make each story's before/after samples comparable.
            torch.manual_seed(2026 + index)
            print(f"\n  Story {index+1}:")
            print(try_prompt(prompt))
    if show_math:
        print("\n➕ MATH (shown as normal arithmetic):")
        for (a, b), prompt in zip(MATH_PROBE_PAIRS, MATH_PROMPTS):
            tokens = tokenizer(prompt, return_tensors="pt").to(DEVICE)
            out = model.generate(**tokens, max_new_tokens=3, do_sample=False,
                                 pad_token_id=tokenizer.eos_token_id)
            continuation = tokenizer.decode(out[0, tokens.input_ids.shape[1]:],
                                            skip_special_tokens=True)
            answer = read_math_answer(continuation)
            if answer is None:
                print(f"  {a} + {b} = ? (raw GPT-2 output: {continuation.strip()!r})")
            else:
                check = "✓" if answer == a + b else f"✗ (expected {a+b})"
                print(f"  {a} + {b} = {answer} {check}")
        print("\nThe notebook translates GPT-2's special training format into ordinary answers.")


## 3. First, look at the outputs **before** you train it! 👀

Here's GPT-2 straight out of the box. What does it do with **three different story openings**? What happens if we ask it to add?

At the end, we'll ask **exactly the same questions**. The three addition problems are kept out of training, so their answers won't be memorized examples. Math will be displayed as ordinary sums—you don't have to decode the model's special training format.

In [ ]:
show_examples("BEFORE TRAINING")

## What do you think? 💭

**Look at those outputs. How do you feel about them?** Charming? Clever? Repetitive? A little ridiculous?

Which stories do you like most? Notice something about the addition answers, too. **Remember your first impression!** These same prompts will help you see what changes.

Before you go on: what do you *predict* will get better? What might get worse? There's no wrong guess here.


## 4. Choose your adventure! 📚 ➕

**Don't like stories? Let's do math!** Choose one track, check the box, and run this cell yourself.

- **Stories:** a quick, tiny first fine-tune.
- **Math:** a longer, more ambitious experiment. GPT-2 practices thousands of additions, and we'll test it on problems it **never saw during training**. Under the hood, we use a digit-ordering trick to make carrying easier; the notebook translates predictions back into normal-looking answers. No backwards-math homework for you!

To try the other track, **restart the runtime** so you begin again with the original GPT-2 weights.

In [ ]:
#@title 🎲 Choose what GPT-2 learns
TRACK = "choose one" #@param ["choose one", "stories", "math"]
IM_READY = False #@param {type:"boolean"}

if TRACK not in ("stories", "math") or not IM_READY:
    raise ValueError("👋 Pick stories or math AND check 'IM_READY', then press ▶ again!")

if TRACK == "stories":
    from datasets import load_dataset
    stream = load_dataset("roneneldan/TinyStories", split="train", streaming=True)
    examples = [row["text"] for row in stream.take(256)]
    PROMPT = "Once upon a time, there was a little dragon"
    print("📚 Stories: 256 examples, 1 epoch. Quick and cozy.")
else:
    rng = random.Random(42)
    # All ordered pairs of 2-digit inputs. Shuffle *before* splitting so
    # evaluation pairs cannot accidentally appear in training.
    all_pairs = [(a, b) for a in range(100) for b in range(100)]
    rng.shuffle(all_pairs)
    train_pairs = all_pairs[:6000]
    test_pairs = all_pairs[6000:6200]  # Held out and never trained on.
    assert all(pair in test_pairs for pair in MATH_PROBE_PAIRS), (
        "Fixed before/after math probes must remain held out of training."
    )

    def math_problem(a, b):
        # Split digits to avoid GPT-2 treating an entire number as one token.
        return f"{' '.join(f'{a:02d}')} + {' '.join(f'{b:02d}')} ="

    def math_answer(a, b):
        # Output ones, tens, hundreds; the carry moves left-to-right.
        # 27 + 58 = 085 -> reversed digits: 5 8 0
        return " ".join(str(a+b).zfill(3)[::-1])

    examples = [math_problem(a,b) + " " + math_answer(a,b) for a,b in train_pairs]
    PROMPT = math_problem(*test_pairs[0])
    print("➕ Math: 6,000 training pairs + 200 completely unseen test pairs.")
    a, b = train_pairs[0]
    print(f"Example: {a} + {b} = {a+b} (shown normally; formatted differently for training).")
    print("This track takes longer than Stories. Choose a T4 GPU!")

## 5. Turn examples into tokens 🔢

We turn examples into tokens for GPT-2. **For math, we pad only to the longest example in our set** instead of making every short addition problem 32 tokens long. Less wasted work, same problems and answers.

Curious about the trick? The model learns to produce the ones digit of the answer first, which makes carrying easier for a left-to-right model. We translate its answers back into ordinary numbers when displaying them.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

MAX_TOKENS = 32 if TRACK == "math" else 128
BATCH_SIZE = 16 if TRACK == "math" else 4
encoded = tokenizer(examples, max_length=MAX_TOKENS, truncation=True,
                    padding="longest", return_tensors="pt")
input_ids = encoded["input_ids"]
attention_mask = encoded["attention_mask"]
labels = input_ids.clone()
labels[attention_mask == 0] = -100
loader = DataLoader(TensorDataset(input_ids, attention_mask, labels),
                    batch_size=BATCH_SIZE, shuffle=True)
print(f"Prepared {len(loader)} batches of up to {BATCH_SIZE} examples.")
print(f"Each example padded to {input_ids.shape[1]} tokens (cap: {MAX_TOKENS}).")
if TRACK == "math":
    check = tokenizer(examples[0], add_special_tokens=False).input_ids
    print("First math sequence token count:", len(check))
    assert len(check) < MAX_TOKENS, "Increase MAX_TOKENS so no math answer is truncated!"

## 6. Train! ⚙️

For each batch: predict next tokens, measure error (**loss**), calculate gradients, update weights.

**Watch the progress bars:** each pass through the training examples gets its own bar. The live **loss** number is the error for the most recent batch (so it may wiggle). After each Math pass, we'll pause to test additions that were **not** in the training data.

**Math gets more practice:** 6,000 different examples over 3 passes. Every epoch, we test a fixed sample of addition problems it has **never seen in training**. This is closer in spirit to successful arithmetic fine-tuning studies: lots of examples, digit-aware formatting, and easier right-to-left carry propagation. It may still struggle, especially on carrying. Accuracy is measured, not promised.

**Stories** stays a one-epoch introduction. Both paths use the *same GPT-2*.

In [ ]:
from torch.optim import AdamW
from tqdm.auto import tqdm

LEARNING_RATE = 1e-4 if TRACK == "math" else 5e-5
EPOCHS = 3 if TRACK == "math" else 1
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
loss_history = []
accuracy_history = []

@torch.no_grad()
def math_accuracy(pairs, batch_size=16):
    """Exact-answer accuracy with genuine greedy autoregressive generation."""
    model.eval()
    old_side = tokenizer.padding_side
    tokenizer.padding_side = "left"  # Required for decoder-only batch generation.
    correct = 0
    for start in range(0, len(pairs), batch_size):
        subset = pairs[start:start + batch_size]
        prompts = [math_problem(a,b) for a,b in subset]
        batch = tokenizer(prompts, return_tensors="pt", padding=True).to(DEVICE)
        generated = model.generate(**batch, do_sample=False, max_new_tokens=3,
                                   pad_token_id=tokenizer.eos_token_id)
        continuations = tokenizer.batch_decode(generated[:, batch.input_ids.shape[1]:],
                                                skip_special_tokens=True)
        for (a,b), output in zip(subset, continuations):
            # Require the model's exact three-digit answer, decoded to a normal sum.
            if read_math_answer(output) == a + b:
                correct += 1
    tokenizer.padding_side = old_side
    model.train()
    return correct / len(pairs)

if TRACK == "math":
    eval_pairs = test_pairs[:64]  # Fixed unseen set; inexpensive progress check.
    print("Checking 64 unseen additions before training…", flush=True)
    accuracy_history.append(math_accuracy(eval_pairs))
    print(f"Before training: {accuracy_history[-1]:.1%} exact addition accuracy on 64 unseen pairs")

model.train()
for epoch in range(EPOCHS):
    with tqdm(loader, desc=f"Pass {epoch+1}/{EPOCHS}", unit="batch") as progress:
        for tokens, mask, targets in progress:
            tokens, mask, targets = (x.to(DEVICE) for x in (tokens, mask, targets))
            optimizer.zero_grad(set_to_none=True)
            loss = model(input_ids=tokens, attention_mask=mask, labels=targets).loss
            loss.backward()
            optimizer.step()
            current_loss = float(loss.item())
            loss_history.append(current_loss)
            progress.set_postfix(loss=f"{current_loss:.3f}", refresh=False)
    if TRACK == "math":
        print(f"Checking 64 held-out additions after pass {epoch+1}…", flush=True)
        accuracy_history.append(math_accuracy(eval_pairs))
        print(f"🔎 Unseen addition accuracy after pass {epoch+1}: {accuracy_history[-1]:.1%}")
print("Finished — you changed a transformer's weights! 🎉")

## 7. What changed? 📉 ✨

Look at loss, and (in Math) **accuracy on problems held out from training**. A falling loss doesn't by itself mean the model knows how to add. Sometimes the best part of science is discovering that your first approach didn't work!

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 3))
plt.plot(loss_history)
plt.xlabel("Training step")
plt.ylabel("Next-token prediction loss")
plt.title(f"GPT-2 learning from {TRACK}")
plt.show()

if TRACK == "math":
    plt.figure(figsize=(7, 3))
    plt.plot(range(len(accuracy_history)), [100*a for a in accuracy_history], marker="o")
    plt.xticks(range(len(accuracy_history)), ["Before"] + [f"Epoch {i}" for i in range(1,EPOCHS+1)])
    plt.ylim(0, 100)
    plt.ylabel("Exact accuracy on 64 unseen sums (%)")
    plt.title("Did it actually learn addition?")
    plt.show()
    print("Checking all 200 held-out additions…", flush=True)
    final_accuracy = math_accuracy(test_pairs)
    print("Final check on 200 held-out examples:", f"{final_accuracy:.1%}")
    print("Six held-out additions, shown in ordinary notation:")
    model.eval()
    for a,b in test_pairs[:6]:
        prompt = math_problem(a,b)
        ins = tokenizer(prompt, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = model.generate(**ins, do_sample=False, max_new_tokens=3,
                                 pad_token_id=tokenizer.eos_token_id)
        prediction = tokenizer.decode(out[0,ins.input_ids.shape[1]:],skip_special_tokens=True)
        answer = read_math_answer(prediction)
        if answer is None:
            print(f"{a} + {b} = ? (expected {a+b}; raw output {prediction.strip()!r})")
        else:
            check = "✓" if answer == a+b else f"✗ (expected {a+b})"
            print(f"{a} + {b} = {answer} {check}")


## 8. Look what you did! 🎉

You trained GPT-2. Now compare **the same prompts as before**, for the track you chose.

- **Stories:** Read all three story openings again. Which changes do you notice? Do you like the stories more now?
- **Math:** See the addition answers, and compare them with the held-out accuracy graph above. Did the model learn a skill it didn't have before?

**Even if an answer is wrong, those weights have changed because of your training.**


In [ ]:
show_examples(
    "AFTER TRAINING — YOU DID THAT!",
    show_stories=(TRACK == "stories"),
    show_math=(TRACK == "math"),
)

## 🎭 One last surprise (Math track only)

You just taught GPT-2 to add. **But what happened to the three stories it could tell earlier?**

Make a prediction before you peek. If you chose **Math**, check the box below and run the cell to reveal the stories *after* math training.

If you chose **Stories**, skip this part—you've already compared your three stories above.


In [ ]:
# @title 🎭 Reveal: what happened to the stories? { display-mode: "form" }
REVEAL_STORIES = False  # @param {type:"boolean"}

if TRACK != "math":
    print("This bonus is for the Math track. Your story comparisons are above!")
elif not REVEAL_STORIES:
    print("What do you predict? Check REVEAL_STORIES, then press ▶ to peek.")
else:
    show_examples("THE SURPRISE — STORIES AFTER LEARNING MATH",
                  show_stories=True, show_math=False)
    print("\n🧠 Compare those stories with the three from BEFORE TRAINING.")
    print("Curious why they may have changed? Open the explanation below.")


<details>
<summary>🧠 Why might the stories have changed? (open after the reveal)</summary>

If learning math made GPT-2 worse at storytelling, that resembles **catastrophic forgetting**: training on one task can interfere with capabilities learned earlier. Three samples are a clue, not proof. How would you test whether it really forgot?

You might be interested in [*Your Model Organisms Might Be Fried*](https://www.lesswrong.com/posts/WmEcgcstzYCcMpc7z/your-model-organisms-might-be-fried) by Daniel Tan and coauthors, about broader ways fine-tuning can degrade model behavior.

</details>


## Now it's yours. 🧭

- **Try the other track:** restart the runtime first, so you start from a fresh GPT-2.
- **Change something:** adjust examples, training length, or learning rate.
- **Math experiments:** Does accuracy differ on sums *with* and *without* carrying? What happens if you change `math_answer` to train on ordinary digit order? What if you give it more training data?
- **Unexpected tradeoffs:** If you tried the Math reveal, did training on sums change the stories? How could you test whether that change holds beyond these three examples?
- **Ask a coding assistant for help** modifying a cell. You don't need permission.
- **Go deeper:** [ARENA](https://arena.education/).

**A note about math:** reversing the digits and adding digit separators are useful training tricks, not magic. Our quick GPT-2 run is *inspired by*, not a reproduction of, published high-accuracy arithmetic experiments. Our actual score is whatever the notebook measures.

**Research to explore:** [number decomposition with fine-tuned GPT-2](https://www.alphaxiv.org/abs/2304.10977) and [How Do Transformers Learn Arithmetic?](https://openreview.net/pdf?id=dsUB4bst9S).

---

<details>
<summary>🗝️ Secret doors: other little worlds to explore (click to open)</summary>

### ♟️ What if a model learned a board game?

**[OthelloGPT](https://github.com/ajyl/mech_int_othelloGPT)** is a transformer trained on sequences of Othello moves. Researchers have studied whether it develops an internal representation of the board.

**Try:** Pick a game position. Can you find out what the model thinks the board looks like? This is a *follow-on project*, not another setting for the GPT-2 training loop.

### ♞ What if the network played chess?

**[Leela Chess Zero](https://github.com/LeelaChessZero/lc0)** is a neural chess engine. The [Leela interpretability project](https://leela-interp.github.io/) is a fascinating entry point into questions about learned look-ahead.

**Try:** Find a position where a move looks surprising, and ask what information the network uses. This one needs a different setup, and its architecture isn't GPT-2.

### 🧩 What if you built a *tiny* transformer yourself?

**Two layers. A tiny vocabulary. A made-up set-substitution rule.**

For example: give it strings containing symbols drawn from `{A, B, C, D}` and define a consistent substitution, such as `A → C, B → D, C → A, D → B`. Train it to transform examples according to the rule. Then change the rule or hold out some combinations.

**Try:** Can it apply the substitution on unseen sequences? Can you inspect what its two layers learn? This is a *project idea*, not a linked pretrained checkpoint.

### 🧭 What next?

You don't need to finish any of these. Pick whatever makes you curious, ask a coding assistant to help you get started, and try something. The only wrong next step is thinking you need permission.

</details>